# Sidorenko's Conjecture

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order

Matrix = np.ndarray

minimize = optimize.minimize




@numba.njit
def count_homomorphisms_mobius_graph_weighted(
    adjacency_matrix: Matrix,
) -> np.float64:
  """Returns the number of homomorphisms from K_{5,5} - C_10 to the input graph.

  A homomorphism from graph G to graph H is a function f: V(G) -> V(H) such that
  for any vertices u and v in G, if u, v are connected then f(u) and f(v) are
  also connected. Note that f is defined on labelled graphs.
  The graph K_{5,5} is the complete bipartite graph with 5 vertices in each
  part. The graph C_10 is a cycle on 10 vertices. And K_{5,5} - C_10 just means
  we start from K_{5,5} and delete a set of edges that make a 10-cycle.

  This function runs an O(n^5) time algorithm but tries taking symmetry into
  account. It was written by Adam Wagner and optimized by us.

  Args:
    adjacency_matrix: the adjacency matrix of the input graph.
  """  # pylint: disable=g-docstring-has-escape
  # TODO(mehrabian): try coding this in C++ to allow for larger sizes
  # TODO(mehrabian): try parallelization for speed up

  n = np.shape(adjacency_matrix)[0]

  assert n <= 85  # if n>85, the answer may not fit in 64 bits (overflow)

  common_neighbours = np.zeros((n, n, n), dtype=np.float64)
  # counts the number of common neighbours of each triple of vertices

  for a in range(n):
    for b in range(a, n):
      for c in range(b, n):
        for d in range(n):
          common_neighbours[a, b, c] += (
              adjacency_matrix[a, d]
              * adjacency_matrix[b, d]
              * adjacency_matrix[c, d]
          )
        common_neighbours[a, c, b] = common_neighbours[a, b, c]
        common_neighbours[b, a, c] = common_neighbours[a, b, c]
        common_neighbours[b, c, a] = common_neighbours[a, b, c]
        common_neighbours[c, a, b] = common_neighbours[a, b, c]
        common_neighbours[c, b, a] = common_neighbours[a, b, c]

  # H = K_{5,5} - C_10 is bipartite: call its parts left part and right part.
  # We do a loop over the ways the left part can be embedded into G.
  # The embedding of this part is described by the 5-tuple v.
  # Since multiple vertices of H can be mapped to same vertex of G, we need to
  # do a case analysis.

  output = np.float64(0)
  old_output = output

  # Case 1: v contains five different vertices. By symmetry, we can assume
  # v0 is strictly smallest, v1 < v4, and multiply the total count by 10
  for v0 in range(n):
    for v1 in range(v0 + 1, n):
      for v4 in range(v1 + 1, n):
        for v2 in range(v0 + 1, n):
          if v2 == v1 or v2 == v4:
            continue
          for v3 in range(v0 + 1, n):
            if v3 != v2 and v3 != v1 and v3 != v4:
              output += np.float64(
                  common_neighbours[v0, v1, v2]
                  * common_neighbours[v1, v2, v3]
                  * common_neighbours[v2, v3, v4]
                  * common_neighbours[v3, v4, v0]
                  * common_neighbours[v4, v0, v1]
                  * 10
              )

  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 2: v[i] = v[i+1] for some i, and other vertices are all different. By
  # symmetry, we assume v0 = v1, v2 < v4, and multiply total count by 10
  for v2 in range(n):
    for v4 in range(v2 + 1, n):
      for v0 in range(n):
        v1 = v0
        if v2 == v0 or v4 == v0:
          continue
        for v3 in range(n):
          if v3 != v0 and v3 != v2 and v3 != v4:
            output += np.float64(
                common_neighbours[v0, v1, v2]
                * common_neighbours[v1, v2, v3]
                * common_neighbours[v2, v3, v4]
                * common_neighbours[v3, v4, v0]
                * common_neighbours[v4, v0, v1]
                * 10
            )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 3: v[i] = v[i+2] for some i, and other vertices are all different. By
  # symmetry, we assume v0 = v2, v3 < v4, and multiply total count by 10
  for v3 in range(n):
    for v4 in range(v3 + 1, n):
      for v0 in range(n):
        v2 = v0
        if v3 == v0 or v4 == v0:
          continue
        for v1 in range(n):
          if v1 != v0 and v1 != v3 and v1 != v4:
            output += np.float64(
                common_neighbours[v0, v1, v2]
                * common_neighbours[v1, v2, v3]
                * common_neighbours[v2, v3, v4]
                * common_neighbours[v3, v4, v0]
                * common_neighbours[v4, v0, v1]
                * 10
            )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 4: v[i] = v[i+1] = v[i+2] for some i, and other vertices are different.
  # By symmetry we assume v0 = v1 = v2, v3 < v4, and multiply by 10.
  for v0 in range(n):
    v1 = v0
    v2 = v0
    for v3 in range(n):
      if v3 == v0:
        continue
      for v4 in range(v3 + 1, n):
        if v4 == v0:
          continue
        output += np.float64(
            common_neighbours[v0, v1, v2]
            * common_neighbours[v1, v2, v3]
            * common_neighbours[v2, v3, v4]
            * common_neighbours[v3, v4, v0]
            * common_neighbours[v4, v0, v1]
            * 10
        )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 5: v[i] = v[i+1] = v[i+3] for some i, and other vertices are different.
  # By symmetry we assume v0 = v1 = v3, v2 < v4, and multiply by 10.
  for v0 in range(n):
    v1 = v0
    v3 = v0
    for v2 in range(n):
      if v2 == v0:
        continue
      for v4 in range(v2 + 1, n):
        if v4 == v0:
          continue
        output += np.float64(
            common_neighbours[v0, v1, v2]
            * common_neighbours[v1, v2, v3]
            * common_neighbours[v2, v3, v4]
            * common_neighbours[v3, v4, v0]
            * common_neighbours[v4, v0, v1]
            * 10
        )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 6: v[i+1] = v[i+2], v[i+3] = v[i+4] for some i, and v[i] is different.
  # By symmetry: v0 != v1 = v2 < v3 = v4 != v0, and multiply by 10.
  for v0 in range(n):
    for v1 in range(n):
      if v1 == v0:
        continue
      v2 = v1
      for v3 in range(v1 + 1, n):
        if v3 == v0:
          continue
        v4 = v3
        output += np.float64(
            common_neighbours[v0, v1, v2]
            * common_neighbours[v1, v2, v3]
            * common_neighbours[v2, v3, v4]
            * common_neighbours[v3, v4, v0]
            * common_neighbours[v4, v0, v1]
            * 10
        )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 7: v[i+1] = v[i+3], v[i+2] = v[i+4] for some i, and v[i] is different.
  # By symmetry: v0 != v1 = v3 < v2 = v4 != v0, and multiply by 10.
  for v0 in range(n):
    for v1 in range(n):
      if v1 == v0:
        continue
      v3 = v1
      for v2 in range(v1 + 1, n):
        if v2 == v0:
          continue
        v4 = v2
        output += np.float64(
            common_neighbours[v0, v1, v2]
            * common_neighbours[v1, v2, v3]
            * common_neighbours[v2, v3, v4]
            * common_neighbours[v3, v4, v0]
            * common_neighbours[v4, v0, v1]
            * 10
        )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 8: v[i+1] = v[i+4], v[i+2] = v[i+3] for some i, and v[i] is different.
  # By symmetry: v1 = v4 != v0, v2 = v3 != v0, and multiply by 5.
  for v0 in range(n):
    for v1 in range(n):
      if v1 == v0:
        continue
      v4 = v1
      for v2 in range(n):
        if v2 == v0 or v2 == v1:
          continue
        v3 = v2
        output += np.float64(
            common_neighbours[v0, v1, v2]
            * common_neighbours[v1, v2, v3]
            * common_neighbours[v2, v3, v4]
            * common_neighbours[v3, v4, v0]
            * common_neighbours[v4, v0, v1]
            * 5
        )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 9: v[i] = v[i+1] = v[i+2] = v[i+3] for some i, and v[i+4] is different.
  # By symmetry: v1=v2=v3=v4 != v0, and multiply the count by 5.
  for v0 in range(n):
    for v1 in range(n):
      if v1 == v0:
        continue
      v2 = v1
      v3 = v1
      v4 = v1
      output += np.float64(
          common_neighbours[v0, v1, v2]
          * common_neighbours[v1, v2, v3]
          * common_neighbours[v2, v3, v4]
          * common_neighbours[v3, v4, v0]
          * common_neighbours[v4, v0, v1]
          * 5
      )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 10: v[i] = v[i+1] != v[i+2] = v[i+3] = v[i+4] for some i. By symmetry,
  # we can assume v0=v1!=v2=v3=v4, and multiply the count by 5
  for v0 in range(n):
    v1 = v0
    for v2 in range(n):
      if v2 == v0:
        continue
      v3 = v2
      v4 = v2
      output += np.float64(
          common_neighbours[v0, v1, v2]
          * common_neighbours[v1, v2, v3]
          * common_neighbours[v2, v3, v4]
          * common_neighbours[v3, v4, v0]
          * common_neighbours[v4, v0, v1]
          * 5
      )
  assert output >= old_output
  old_output = output

  # Case 11: v[i] = v[i+2] != v[i+1] = v[i+3] = v[i+4] for some i. By symmetry,
  # we can assume v0=v2!=v1=v3=v4, and multiply the count by 5
  for v0 in range(n):
    v2 = v0
    for v1 in range(n):
      if v1 == v0:
        continue
      v3 = v1
      v4 = v1
      output += np.float64(
          common_neighbours[v0, v1, v2]
          * common_neighbours[v1, v2, v3]
          * common_neighbours[v2, v3, v4]
          * common_neighbours[v3, v4, v0]
          * common_neighbours[v4, v0, v1]
          * 5
      )
  assert output >= old_output  # sanity check to avoid overflow
  old_output = output

  # Case 12: all v[i] are equal to some vertex u
  for u in range(n):
    output += np.float64(common_neighbours[u, u, u] ** 5)
  assert output >= old_output  # sanity check to avoid overflow

  return output


def score_weighted(adjacency_matrix: Matrix) -> np.double:
  """Weighted score function for Sidorenko conjecture for Mobius graph.

  Let H = K_{5,5} - C_10. For input weighted graph G (with adjacency matrix),
  this function computes:
  (2 * weighted_edges(G))^15 / (v(G)^20 * hom(H, G)) - 1
  where v(G) is the number of vertices, weighted_edges(G) is the sum of edge
  weights, and hom(H, G) is the weighted homomorphism count from H to G.
  A positive score would suggest a counterexample to (weighted) Sidorenko's
  conjecture.

  Args:
    adjacency_matrix: Weighted adjacency matrix of the graph.

  Returns:
    A score in [-1, infinity). Returns -1 if hom(H, G) = 0 or no edges.
  """

  vertices: int = int(np.shape(adjacency_matrix)[0])

  if vertices != 30:
    return -1.0

  if vertices == 0:
    return -1.0

  # clip all the weights to be in [0, 1]
  adjacency_matrix = np.clip(adjacency_matrix, 0, 1)

  # make sure that the adjacency matrix is symmetric
  adjacency_matrix = (adjacency_matrix + adjacency_matrix.T) / 2

  # Scale it so the mean is 0.5
  adjacency_matrix = adjacency_matrix / (2 * adjacency_matrix.mean())
  adjacency_matrix = np.clip(adjacency_matrix, 0, 1)

  # If the matrix is near-constant, return -1
  if np.max(adjacency_matrix) - np.min(adjacency_matrix) < 0.25:
    return -1.0

  # If the standard deviation of the matrix is small, return -1
  if np.std(adjacency_matrix) < 0.23:
    return -1.0

  weighted_twice_edges: float = np.sum(adjacency_matrix)  # Sum of all weights
  if weighted_twice_edges == 0:
    return -1.0

  homomorphisms: float = count_homomorphisms_mobius_graph_weighted(
      adjacency_matrix
  )
  if homomorphisms == 0:
    return -1.0

  numerator_weighted: float = (weighted_twice_edges**15) - (
      vertices**20
  ) * homomorphisms
  denominator_weighted: float = (vertices**20) * homomorphisms

  return np.double(numerator_weighted / denominator_weighted)


def score_weighted_h(adjacency_matrix: Matrix) -> np.double:
  """Weighted score function for Sidorenko conjecture for Mobius graph.

  Let H = K_{5,5} - C_10. For input weighted graph G (with adjacency matrix),
  this function computes:
  (2 * weighted_edges(G))^15 / (v(G)^20 * hom(H, G)) - 1
  where v(G) is the number of vertices, weighted_edges(G) is the sum of edge
  weights, and hom(H, G) is the weighted homomorphism count from H to G.
  A positive score would suggest a counterexample to (weighted) Sidorenko's
  conjecture.

  Args:
    adjacency_matrix: Weighted adjacency matrix of the graph.

  Returns:
    A score in [-1, infinity). Returns -1 if hom(H, G) = 0 or no edges.
  """
  vertices: int = int(np.shape(adjacency_matrix)[0])
  if vertices == 0:
    return -1.0

  if vertices != 30:
    return -1.0

  # clip all the weights to be in [0, 1]
  adjacency_matrix = np.clip(adjacency_matrix, 0, 1)

  # make sure that the adjacency matrix is symmetric
  adjacency_matrix = (adjacency_matrix + adjacency_matrix.T) / 2

  # Scale it so the mean is 0.5
  # First check if dividing by zero
  if adjacency_matrix.mean() < 0.01:
    return -1.0
  adjacency_matrix = adjacency_matrix / (2 * adjacency_matrix.mean())
  adjacency_matrix = np.clip(adjacency_matrix, 0, 1)

  # If the matrix is near-constant, return -1
  if np.max(adjacency_matrix) - np.min(adjacency_matrix) < 0.25:
    return -1.0

  # If the standard deviation of the matrix is small, return -1
  if np.std(adjacency_matrix) < 0.23:
    return -1.0

  weighted_twice_edges: float = np.sum(adjacency_matrix)  # Sum of all weights
  if weighted_twice_edges == 0:
    return -1.0

  homomorphisms: float = count_homomorphisms_mobius_graph_weighted(
      adjacency_matrix
  )
  if homomorphisms == 0:
    return -1.0

  numerator_weighted: float = (weighted_twice_edges**15) - (
      vertices**20
  ) * homomorphisms
  denominator_weighted: float = (vertices**20) * homomorphisms

  return np.double(numerator_weighted / denominator_weighted)


def format_feedback_repr(feedback: dict[str, Any]):
  """Formats feedback dictionary for better representation of numpy arrays.

  Args:
    feedback: A dictionary containing feedback information.

  Returns:
    A dictionary with formatted feedback, where numpy arrays are represented
    as strings that can be directly used to reconstruct the arrays.
  """
  formatted_feedback = {}
  with np.printoptions(threshold=np.inf, linewidth=np.inf):
    for key, value in feedback.items():
      if isinstance(value, np.ndarray):
        repr_str = repr(
            value
        )  # Get repr string (e.g., "array([[...], [...]])")
        cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

        # Remove the leading "array(" and trailing ")" from repr string, then
        # wrap with "np.array(...)"
        array_content = cleaned_repr_str[
            6:-1
        ]  # Extract content inside "array(...)"

        if np.iscomplexobj(value):
          formatted_feedback[key] = (  # Use extracted content in np.array
              f'np.array({array_content}, dtype=np.complex128)'
          )
        elif not np.issubdtype(value.dtype, np.inexact):
          formatted_feedback[key] = (
              f'np.array({array_content}, dtype=np.float64)'
          )
        else:
          formatted_feedback[key] = f'np.array({array_content})'

      else:
        formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the numerical bound for the polygons if valid, or 0 if invalid."""
  result = {}
  feedback = {}
  del hypers
  best_construction, advice = search_for_best_adjacency_matrix()
  best_construction = np.clip(best_construction, 0, 1)

  # make sure that the adjacency matrix is symmetric
  best_construction = (best_construction + best_construction.T) / 2
  result['score'] = float(score_weighted_h(best_construction))

  feedback['best_adjacency_matrix'] = best_construction
  feedback['best_score_found'] = result['score']
  feedback['advice'] = advice
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds a function that give the best bound for Sendov's conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import re
from typing import Any, Callable, Mapping, Tuple
import scipy.linalg as la
import numpy.polynomial.polynomial as poly
import numba



def search_for_best_adjacency_matrix() -> Tuple[np.ndarray, str]:
  """Function to search for the best 30x30 graphon construction."""
  vertex_count = 30  # DON'T CHANGE THIS NUMBER
  adjacency_matrix = np.random.uniform(0, 1, (vertex_count, vertex_count))

  advice = ''
  best_adjacency_matrix = adjacency_matrix.copy()
  best_score = score_weighted(adjacency_matrix)
  start_time = time.time()
  eval_count = 0
  while time.time() - start_time < np.random.randint(10, 500):
    adjacency_matrix[
        np.random.randint(10), np.random.randint(10)
    ] += np.random.uniform(-0.1, 0.1)
    # score_time_start = time.time()
    score = score_weighted(adjacency_matrix)
    if score < -0.9:
      adjacency_matrix = np.random.uniform(0, 1, (vertex_count, vertex_count))
      score = score_weighted(adjacency_matrix)
    # print(f"time: {time.time() - score_time_start}")
    if score > best_score:
      best_score = score
      best_adjacency_matrix = adjacency_matrix.copy()
      logging.info(best_score)
    eval_count += 1
  advice += (
      f'Evaluated {eval_count} graphs with {vertex_count} vertices within'
      f' {int(time.time() - start_time)} seconds. My advice for future'
      ' generations is to stay away from the constant matrix, and to try to'
      ' keep all values between 0 and 1 but with a high variance. Good luck'
      " guys, we've got this!"
  )
  logging.info(eval_count)
  return best_adjacency_matrix, advice

**Prompt used**

Act as an expert software developer and graph theorist specializing in finding
counterexamples to Sidorenko's conjecture. You will be trying to come up with a
counterexample to this conjecture, that is, a graphon G such that a particular
bipartite graph H has few homomorphisms into it relative to its density. The
graph H will be the Mobius graph, that is fixed. Your task is to find a good
candidate for G. All entries of the adjacency matrix of G will have to be
values between 0 and 1 (as G is a graphon).

As a reminder, this is the function you have to optimize:

def score_weighted(adjacency_matrix: Matrix) -> np.double:
  """Weighted score function for Sidorenko conjecture for Mobius graph.

Let H = K_(5,5) - C_10. For input weighted graph G (with adjacency matrix),
  this function computes:
  (2 * weighted_edges(G))^15 / (v(G)^20 * hom(H, G)) - 1
  where v(G) is the number of vertices, weighted_edges(G) is the sum of edge
  weights, and hom(H, G) is the weighted homomorphism count from H to G.
  A positive score would suggest a counterexample to (weighted) Sidorenko's
  conjecture.

Args:
    adjacency_matrix: Weighted adjacency matrix of the graph.

Returns:
    A score in [-1, infinity). Returns -1 if hom(H, G) = 0 or no edges.
  """

vertices: int = int(np.shape(adjacency_matrix)[0])
  if vertices != 30:
    return -1.0

# clip all the weights to be in [0, 1]
  adjacency_matrix = np.clip(adjacency_matrix, 0, 1)

# make sure that the adjacency matrix is symmetric
  adjacency_matrix = (adjacency_matrix + adjacency_matrix.T) / 2

# If the matrix is near-constant, return -1
  if np.max(adjacency_matrix) - np.min(adjacency_matrix) < 0.1:
    return -1.0

# If the standard deviation of the matrix is small, return -1
  if np.std(adjacency_matrix) < 0.1:
    return -1.0

weighted_twice_edges: float = np.sum(adjacency_matrix)  # Sum of all weights
  if weighted_twice_edges == 0:
    return -1.0

homomorphisms: float = count_homomorphisms_mobius_graph_weighted(
      adjacency_matrix
  )
  if homomorphisms == 0:
    return -1.0

numerator_weighted: float = (weighted_twice_edges15) - (
      vertices20
  ) * homomorphisms
  denominator_weighted: float = (vertices**20) * homomorphisms

return np.double(numerator_weighted / denominator_weighted)

Your task is to write a randomized search function that searches for the best
adjacency matrix. Your function will have 1000 seconds to run, and after that
it has to have returned the best construction it found. If after 1000 seconds
it has not returned anything, it will be terminated with negative infinity
points. You can use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to
define the "start_time" variable early in your program. The number of vertices,
n, will be fixed at 30 in this experiment.

You may code up any search method you want, and you are allowed to call the
score_weighted() function as many times as you want. You have access to it, you
don't need to code up the score_weighted() function.

Your function will also return an "advice" string. You can put into this
anything you want! Future generations will see this advice, and may follow it if
they choose to. I would recommend you put into here anything you have discovered
in your 1000 seconds -- things that seemed to work well, things that didn't work
well, what you would have tried if you had more time, some creative encouraging
words, etc. If you have no clue how to get a counterexample, feel free to say
so in the "advice", honesty is much preferred over false leads.

Constant matrices would receive a good score, but they are local optima and
clearly not interesting. Because of this, your constructions receive a penalty
if they are too close to the constant function, as you can see in the
score_weighted() function above. The counterexample will have to look very
different from a constant matrix.

Unfortunately we don't have a good idea what the counterexample will look like,
only that standard optimization methods have not been able to find it so far. We
know the counterexample will NOT be a nice, structured, regular object. So do
not hesitate to try unconventional and randomized methods, this problem is going
to be all about exploring novel ideas and trying enough random constructions
until we get lucky. Have fun!

## What AlphaEvolve found

The smallest bipartite graph for which the Sidorenko property is not known to hold is the graph obtained by removing a 10-cycle from $K_{5,5}$. AlphaEvolve was used to search for a graphon $W$ which violates Sidorenko's inequality for this graph. Despite various attempts, including adding penalties for graphons close to being constant, AlphaEvolve did not manage to find a counterexample to this conjecture.